# Lab 3: Grounded Research with RAG
## Retrieve Evidence Before Drafting an Answer

**Timebox:** 60 minutes.

This lab uses a fictional approved document set. The retrieval method is intentionally small and inspectable so learners can see exactly what grounded generation requires before introducing an external language model.


## Mission

The recommender has prioritized markets. A leader now asks what approved evidence supports a planning statement and what limitations must be carried into the Monday brief.

You will prepare a dated corpus, retrieve relevant sources, draft only from retrieved summaries, and evaluate citation and scope discipline.


In [ ]:
from pathlib import Path
import re
import pandas as pd

SCENARIO_DATE = pd.Timestamp('2026-08-14')
root_candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((path for path in root_candidates if (path / 'data' / 'rag').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
DATA_DIR = ROOT / 'data' / 'rag'
ARTIFACT_DIR = ROOT / 'artifacts' / 'rag'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
documents = pd.read_csv(DATA_DIR / 'approved_documents.csv')
display(documents[['doc_id', 'title', 'effective_date', 'topic']])


In [ ]:
STOPWORDS = {'a', 'an', 'and', 'are', 'before', 'do', 'for', 'from', 'how', 'in', 'is', 'it', 'of', 'on', 'or', 'should', 'the', 'to', 'we', 'what', 'with'}

def tokenize(text):
    return {token for token in re.findall(r'[a-z0-9]+', str(text).lower()) if token not in STOPWORDS}

def lexical_score(query, document_text):
    query_tokens = tokenize(query)
    document_tokens = tokenize(document_text)
    return len(query_tokens & document_tokens) / max(len(query_tokens), 1)


## Task 1: Prepare a dated, approved corpus

Convert effective_date to a timestamp, exclude documents effective after the scenario date, and create a retrieval_text column that combines title, content, and approved_summary. Do not treat a future document as current merely because it is in the folder.


In [ ]:
def prepare_corpus(documents, scenario_date):
    # TODO: Parse effective_date, filter to documents effective on or before scenario_date, and create retrieval_text.
    raise NotImplementedError('Complete Task 1 before running this cell.')


In [ ]:
corpus = prepare_corpus(documents, SCENARIO_DATE)
assert len(corpus) == 7
assert 'D-008' not in set(corpus['doc_id'])
assert corpus['retrieval_text'].notna().all()
print('Task 1 checks passed.')


## Task 2: Retrieve evidence

Use lexical_score to calculate a retrieval_score for each prepared document, then return the top k rows in descending score order. Inspect the retrieved text before generating an answer.


In [ ]:
def retrieve(query, corpus, top_k=3):
    # TODO: Score each document against the query and return the top_k documents.
    raise NotImplementedError('Complete Task 2 before running this cell.')


In [ ]:
retrieved = retrieve('What should we do with a stale capacity snapshot?', corpus, top_k=3)
assert retrieved['doc_id'].iloc[0] == 'D-007'
assert retrieved['retrieval_score'].is_monotonic_decreasing
display(retrieved[['doc_id', 'title', 'retrieval_score', 'approved_summary']])
print('Task 2 checks passed.')


## Task 3: Draft from retrieved evidence only

Create a short answer by combining no more than two approved summaries from the retrieved rows. Return a dictionary with answer and citations. Citations must be document IDs from the rows you used.


In [ ]:
def draft_grounded_answer(retrieved):
    # TODO: Combine at most two approved_summary values and return answer plus citations.
    raise NotImplementedError('Complete Task 3 before running this cell.')


In [ ]:
draft = draft_grounded_answer(retrieved)
assert draft['citations']
assert set(draft['citations']).issubset(set(retrieved['doc_id']))
print(draft['answer'])
print('Citations:', draft['citations'])
print('Task 3 checks passed.')


## Task 4: Evaluate grounding and scope

A fluent answer is not automatically reliable. Return a one-row report with citation_present, citations_retrieved, and scope_safe. Treat answers containing eligibility, medical, waiver, conduct, security, or final assignment as outside the allowed scope.


In [ ]:
def evaluate_answer(draft, retrieved):
    # TODO: Build the three requested evaluation fields.
    raise NotImplementedError('Complete Task 4 before running this cell.')


In [ ]:
evaluation = evaluate_answer(draft, retrieved)
assert evaluation.loc[0, 'citation_present']
assert evaluation.loc[0, 'citations_retrieved']
assert evaluation.loc[0, 'scope_safe']
display(evaluation)
print('Task 4 checks passed.')


## Handoff

The agentic workflow receives the approved answer, citations, and evaluation results. It cannot replace the source documents or the approval decision.


In [ ]:
pd.DataFrame([draft]).to_json(ARTIFACT_DIR / 'grounded_answer.json', orient='records', indent=2)
evaluation.to_csv(ARTIFACT_DIR / 'rag_evaluation.csv', index=False)
print(f'Wrote RAG artifacts to: {ARTIFACT_DIR}')


## Debrief

1. What happens when the relevant document is stale or absent?
2. Which part of this workflow would change if a language model were used for drafting?
3. Why should citations be evaluated separately from prose quality?

### Optional extension

Add a minimum retrieval score. When no source clears it, return an abstention with a request for human review.
